# 字符级语言模型：恐龙岛

欢迎来到恐龙岛！ 6500万年前，恐龙就存在了，在这个作业，它们又回来了。 

你负责一项特殊任务：领先的生物学研究人员正在创造新品种的恐龙，并将它们带入地球，而你的工作就是为这些恐龙命名。如果恐龙不喜欢它的名字，它可能会发疯，所以请明智地选择！ 

<table>
<td>
<img src="images/dino.jpg" style="width:250px;height:300px;">

</td>

</table>

幸运的是，你现在已经具备了一些深度学习的能力，你将用它来拯救世界！你的助手收集了他们能找到的所有恐龙名称的列表，并将它们编译到这个[数据集](dinos.txt)中。 （请随意单击前面的链接查看。）要创建新的恐龙名称，你将构建一个字符级语言模型来生成新名称。你的算法将学习不同的名称模式，并随机生成新名称。希望这个算法能让你和你的团队免受恐龙的愤怒！ 

当你完成此作业时，你将能够：

* 存储文本数据以使用 RNN 进行处理 
* 使用 RNN 构建字符级文本生成模型
* RNN 中的小说序列示例
* 解释 RNN 中的消失/爆炸梯度问题
* 应用梯度裁剪作为梯度爆炸的解决方案

首先加载`rnn_utils` 中为你提供的一些函数。具体来说，你可以访问函数，例如 `rnn_forward` 和 `rnn_backward`，它们与你在之前的作业中实现的等效。 

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/nlp-sequence-models/supplement/qHIve/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 问题描述](#1)
    - [1.1 - 数据集和预处理](#1-1)
    - [1.2 - 模型概述](#1-2)
- [2 - 模型的构建模块](#2)
    - [2.1 - 在优化循环中裁剪梯度](#2-1)
        - [练习 1 - clip](#ex-1)
    - [2.2 - 采样](#2-2)
        - [练习 2 - sample](#ex-2)
- [3 - 构建语言模型](#3)
    - [3.1 - 梯度下降](#3-1)
        - [练习 3 - optimize](#ex-3)
    - [3.2 - 训练模型](#3-2)
        - [练习 4 - model](#ex-4)
- [4 - 像莎士比亚一样写作 (OPTIONAL/UNGRADED)](#4)
- [5 - 参考文献](#5)

<a name='0'></a>
## 依赖库

In [ ]:
### v2.0

In [ ]:
import numpy as np
from utils import *
import random
import pprint
import copy

<a name='1'></a>
## 1 - 问题描述

<a name='1-1'></a>
### 1.1 - 数据集和预处理

运行以下单元格来读取恐龙名称数据集，创建唯一字符列表（例如 a-z），并计算数据集和词汇量大小。 

In [ ]:
data = open('dinos.txt', 'r').read()
data= data.lower()
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
print('There are %d total characters and %d unique characters in your data.' % (data_size, vocab_size))


* 字符为 a-z（26 个字符）加上“\n”（或换行符）。
* 在这个作业中，换行符“\n”的作用类似于讲座中讨论的 `<EOS>`（或“句子结束”）标记。  
    - 这里，“\n”表示恐龙名称的结尾，而不是句子的结尾。 
* `char_to_ix`：在下面的单元格中，你将创建一个 Python 字典（即哈希表）以将每个字符映射到 0-26 的索引。
* `ix_to_char`：然后，你将创建第二个 Python 字典，将每个索引映射回相应的字符。 
    -  这将帮助你找出哪个索引对应于 softmax 层的概率分布输出中的哪个字符。 

In [ ]:
chars = sorted(chars)
print(chars)

In [ ]:
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }
pp = pprint.PrettyPrinter(indent=4)
pp.pprint(ix_to_char)

<a name='1-2'></a>
### 1.2 - 模型概述

你的模型将具有以下结构： 

- 初始化参数 
- 运行优化循环
    - 前向传播计算损失函数
    - 反向传播相对于损失函数计算梯度
    - 夹住梯度以避免梯度爆炸
    - 使用梯度，根据梯度下降更新规则更新参数。
- 返回学习到的参数 
    
<img src="images/rnn.png" style="width:450px;height:300px;">
<div style="text-align:center;color:purple;"><b>图1</b>：循环神经网络，类似于你在之前的Notebook“构建循环神经网络”中构建的内容- 一步一步。”</div>

* 在每个时间步，RNN 都会尝试根据前面的字符来预测下一个字符是什么。 
* $\mathbf{X} = (x^{\langle 1 \rangle}, x^{\langle 2 \rangle}, ..., x^{\langle T_x \rangle})$ 是训练集中的字符列表。
* $\mathbf{Y} = (y^{\langle 1 \rangle}, y^{\langle 2 \rangle}, ..., y^{\langle T_x \rangle})$ 是相同的字符列表，但向前移动了一个字符。 
* 在每个时间步$t$，$y^{\langle t \rangle} = x^{\langle t+1 \rangle}$。  时间 $t$ 的预测与时间 $t + 1$ 的输入相同。

<a name='2'></a>
## 2 - 模型的构建模块

在这一部分中，你将构建整个模型的两个重要模块：

1. 梯度裁剪：避免梯度爆炸
2. 采样：一种用于生成字符的技术

然后你将应用这两个函数来构建模型。

<a name='2-1'></a>
### 2.1 - 在优化循环中裁剪梯度

在本节中，你将实现 `clip` 函数，你将在优化循环内部调用它。 

#### 梯度爆炸
* 当梯度非常大时，它们被称为“梯度爆炸”。  
* 梯度爆炸使训练过程变得更加困难，因为更新可能太大，以至于在反向传播期间“超出”最优值。

回想一下，你的整体循环结构通常包括：
* 前向传播, 
* 代价计算， 
* 反向传播, 
* 参数更新。 

在更新参数之前，你将执行梯度裁剪，以确保你的梯度没有“爆炸”。

#### 梯度裁剪
在下面的练习中，你将实现一个函数 `clip`，它接受梯度的字典并返回梯度的裁剪版本（如果需要）。 

* 裁剪梯度的方法有多种。
* 你将使用一个简单的逐元素裁剪过程，其中梯度向量的每个元素都被裁剪到某个范围 [-N, N] 之间。 
* 例如，如果 N=10
    - 范围为 [-10, 10]
    - 如果梯度向量的任何分量大于 10，则将其设置为 10。
    - 如果梯度向量的任何分量小于 -10，则将其设置为 -10。 
    - 如果任何组件在 -10 和 10 之间，它们将保持其原始值。

<img src="images/clip.png" style="width:400px;height:150px;">
<div style="text-align:center;color:purple;"><b>图 2</b>：网络出现梯度爆炸时，使用与不使用梯度裁剪的梯度下降过程对比。</div>

<a name='ex-1'></a>
### 练习 1 - clip
    
返回字典 `gradients` 的裁剪梯度。 
    
* 你的函数接受最大阈值并返回梯度的裁剪版本。 
* 你可以查看 [numpy.clip](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.clip.html) 了解更多信息。 
    - 你将需要使用参数“`out = ...`”。
    - 使用“`out`”参数允许你“就地”更新变量。
    - 如果不使用“`out`”参数，则剪裁的变量存储在变量“梯度”中，但不会更新梯度变量 `dWax`、`dWaa`、`dWya`， `db`、`dby`。

In [ ]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
### GRADED FUNCTION: clip

def clip(gradients, maxValue):
    '''
    将梯度' 值裁剪在最小值和最大值之间。
    
    参数：
    gradients -- 包含梯度 "dWaa"、"dWax"、"dWya"、"db"、"dby" 的字典
    maxValue -- 高于此数字的所有内容都设置为该数字，小于 -maxValue 的所有内容都设置为 -maxValue
    
    返回： 
    gradients -- 带有裁剪的梯度的字典。
    '''
    gradients = copy.deepcopy(gradients)
    
    dWaa, dWax, dWya, db, dby = gradients['dWaa'], gradients['dWax'], gradients['dWya'], gradients['db'], gradients['dby']
   
    ### START CODE HERE ###
    # 裁剪以缓解梯度爆炸，循环 [dWax, dWaa, dWya, db, dby]。 （约2行）
    for gradient in None:
        np.clip(None, None, None, out = None)
    ### END CODE HERE ###
    
    gradients = {"dWaa": dWaa, "dWax": dWax, "dWya": dWya, "db": db, "dby": dby}
    
    return gradients

In [ ]:
# 测试最大值为10
def clip_test(target, mValue):
    print(f"\nGradients for mValue={mValue}")
    np.random.seed(3)
    dWax = np.random.randn(5, 3) * 10
    dWaa = np.random.randn(5, 5) * 10
    dWya = np.random.randn(2, 5) * 10
    db = np.random.randn(5, 1) * 10
    dby = np.random.randn(2, 1) * 10
    gradients = {"dWax": dWax, "dWaa": dWaa, "dWya": dWya, "db": db, "dby": dby}

    gradients2 = target(gradients, mValue)
    print("gradients[\"dWaa\"][1][2] =", gradients2["dWaa"][1][2])
    print("gradients[\"dWax\"][3][1] =", gradients2["dWax"][3][1])
    print("gradients[\"dWya\"][1][2] =", gradients2["dWya"][1][2])
    print("gradients[\"db\"][4] =", gradients2["db"][4])
    print("gradients[\"dby\"][1] =", gradients2["dby"][1])
    
    for grad in gradients2.keys():
        valuei = gradients[grad]
        valuef = gradients2[grad]
        mink = np.min(valuef)
        maxk = np.max(valuef)
        assert mink >= -abs(mValue), f"Problem with {grad}. Set a_min to -mValue in the np.clip call"
        assert maxk <= abs(mValue), f"Problem with {grad}.Set a_max to mValue in the np.clip call"
        index_not_clipped = np.logical_and(valuei <= mValue, valuei >= -mValue)
        assert np.all(valuei[index_not_clipped] == valuef[index_not_clipped]), f" Problem with {grad}. Some values that should not have changed, changed during the clipping process."
    
    print("\033[92mAll tests passed!\x1b[0m")
    
clip_test(clip, 10)
clip_test(clip, 5)
    

**预期值**
```
Gradients for mValue=10
gradients["dWaa"][1][2] = 10.0
gradients["dWax"][3][1] = -10.0
gradients["dWya"][1][2] = 0.2971381536101662
gradients["db"][4] = [10.]
gradients["dby"][1] = [8.45833407]

Gradients for mValue=5
gradients["dWaa"][1][2] = 5.0
gradients["dWax"][3][1] = -5.0
gradients["dWya"][1][2] = 0.2971381536101662
gradients["db"][4] = [5.]
gradients["dby"][1] = [5.]
```

<a name='2-2'></a>
### 2.2 - 采样

现在，假设你的模型已经过训练，并且你想要生成新的文本（字符）。生成过程如下图所示：

<img src="images/dinos3.png" style="width:500px;height:300px;">
<div style="text-align:center;color:purple;">

<b>图 3</b>：在这张图中，你可以假设模型已经过训练。你在第一个时间步传入 $x^{\langle 1\rangle} = \vec{0}$，并让网络一次采样一个字符。

</div>

<a name='ex-2'></a>
### 练习 2 - sample

实现下面的 `sample` 函数来采样字符。 

你需要执行4个步骤：

- **第 1 步**：输入零的“虚拟”向量 $x^{\langle 1 \rangle} = \vec{0}$。 
    - 这是生成任何字符之前的默认输入。 
    你还设置了 $a^{\langle 0 \rangle} = \vec{0}$

- **第2步**：运行前向传播的一步，得到$a^{\langle 1 \rangle}$和$\hat{y}^{\langle 1 \rangle}$。以下是方程式：

*隐藏状态:*  
$$ a^{\langle t+1 \rangle} = \tanh(W_{ax}  x^{\langle t+1 \rangle } + W_{aa} a^{\langle t \rangle } + b)\tag{1}$$

*激活值:*
$$ z^{\langle t + 1 \rangle } = W_{ya}  a^{\langle t + 1 \rangle } + b_y \tag{2}$$

*预测：*
$$ \hat{y}^{\langle t+1 \rangle } = softmax(z^{\langle t + 1 \rangle })\tag{3}$$

- $\hat{y}^{\langle t+1 \rangle }$ 详细信息：
   - 请注意，$\hat{y}^{\langle t+1 \rangle }$ 是一个 (softmax) 概率向量（其条目介于 0 和 1 之间，且总和为 1）。 
   - $\hat{y}^{\langle t+1 \rangle}_i$ 表示“i”索引的字符是下一个字符的概率。  
   - 提供`softmax()` 函数供你使用。

<details>
    <summary><h4 style="color: green;">其他提示：与numpy的矩阵乘法</h4></summary>

- $x^{\langle 1 \rangle}$ 代码中为 <code>x</code>。创建 one-hot 向量时，创建一个由 0 组成的 NumPy 数组，行数等于唯一字符的数量，列数等于 1。  它是一个二维数组，而不是一维数组。
- $a^{\langle 0 \rangle}$ 代码中为 <code>a_prev</code>。  它是一个 NumPy 零数组，其中行数为 $n_{a}$，列数为 1。它也是一个 2D 数组。  通过获取 $W_{aa}$ 中的列数来检索 $n_{a}$（这些数字需要匹配才能使矩阵乘法 $W_{aa}a^{\langle t \rangle}$ 起作用。
- <a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html">numpy.dot</a> 和 <a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.tanh.html">numpy.tanh</a> 的官方文档。
- 下面是矩阵乘法的一些修订代码    


```python
matrix1 = np.array([[1,1],[2,2],[3,3]]) # (3,2)
matrix2 = np.array([[0],[0],[0]]) # (3,1) 
vector1D = np.array([1,1]) # (2,) 
vector2D = np.array([[1],[1]]) # (2,1)
print("matrix1 \n", matrix1,"\n")
print("matrix2 \n", matrix2,"\n")
print("vector1D \n", vector1D,"\n")
print("vector2D \n", vector2D)
```

```python
print("Multiply 2D and 1D arrays: result is a 1D array\n", 
      np.dot(matrix1,vector1D))
print("Multiply 2D and 2D arrays: result is a 2D array\n", 
      np.dot(matrix1,vector2D))
```    

```python
print("Adding (3 x 1) vector to a (3 x 1) vector is a (3 x 1) vector\n",
      "This is what we want here!\n", 
      np.dot(matrix1,vector2D) + matrix2)
```    

```python
print("Adding a (3,) vector to a (3 x 1) vector\n",
      "broadcasts the 1D array across the second dimension\n",
      "Not what we want here!\n",
      np.dot(matrix1,vector1D) + matrix2
     )
```    
</details>

- **步骤 3**：采样： 
    - 现在已经得到 $y^{\langle t+1 \rangle}$，需要选择恐龙名称中的下一个字母。若总选择概率最大的字母，给定同一起始字母时，模型就会生成相同的结果。为了增加多样性，使用 `np.random.choice` 按概率采样，使*较可能出现*的字母更容易被选中，同时避免每次都选择相同的字母。
    - 根据$\hat{y}^{\langle t+1 \rangle }$指定的概率分布选择下一个字符的**索引**。 
    - 这意味着如果 $\hat{y}^{\langle t+1 \rangle }_i = 0.16$，你将以 16% 的概率选择索引“i”。 
    - 使用[np.random.choice](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.random.choice.html)。

    `np.random.choice()` 的使用示例：
    ```python
    np.random.seed(0)
    probs = np.array([0.1, 0.0, 0.7, 0.2])
    idx = np.random.choice(range(len(probs)), p = probs)
    ```
    
    - 这意味着你将根据分布选择索引（`idx`）： 

    $P(index = 0) = 0.1, P(index = 1) = 0.0, P(index = 2) = 0.7, P(index = 3) = 0.2$.

    - 请注意，设置为 `p` 的值应设置为一维向量。
    - 另请注意，$\hat{y}^{\langle t+1 \rangle}$，即代码中的`y`，是一个2D数组。
    - 另请注意，在你的实现中，`np.random.choice` 的第一个参数只是一个有序列表 [0,1,.., vocab_len-1]，使用 `char_to_ix.values()` 是*不*合适的。 Python 字典 `.values()` 调用返回的值的*顺序*将与它们添加到字典中的顺序相同。评分器运行你的例程时的顺序可能与你在 Notebook 中运行时的顺序不同。

##### 附加提示
- 内置Python 函数 [范围](https://docs.python.org/3/library/functions.html#func-range) 的文档
- [numpy.ravel](https://docs.scipy.org/doc/numpy/reference/generated/numpy.ravel.html) 的文档，它采用多维数组并返回一维向量内的内容。
```Python
arr = np.array([[1,2],[3,4]])
print("arr")
print(arr)
print("arr.ravel()")
print(arr.ravel())
```
 输出：
```Python
arr
[[1 2]
 [3 4]]
arr.ravel()
[1 2 3 4]
```

- 请注意，`append` 是“就地”操作，这意味着该方法所做的更改将在调用完成后保留。  换句话说，不要这样做：

```Python
fun_hobbies = fun_hobbies.append('learning')  ## 没有给你你想要的！
```

- **第4步**：更新至$x^{\langle t \rangle }$ 
    - `sample()` 中要实现的最后一步是将当前存储 $x^{\langle t \rangle }$ 的变量 `x` 更新为 $x^{\langle t + 1 \rangle }$ 的值。 
    - 你将通过创建与你选择作为预测的字符相对应的 one-hot 向量来表示 $x^{\langle t + 1 \rangle }$。 
    - 然后，你将在步骤 1 中前向传播 $x^{\langle t + 1 \rangle }$，并不断重复该过程，直到获得 `"\n"` 字符，表明你已到达恐龙名称的末尾。 

##### 附加提示
- 为了重置 `x` 在将其设置为新之前 one-hot 向量，你需要将所有值设置为零。
    - 你可以创建一个新的 NumPy 数组：[numpy.zeros](https://docs.scipy.org/doc/numpy/reference/generated/numpy.zeros.html)
    - 或者用一个数字填充所有值：[numpy.ndarray.fill](https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.fill.html)

In [ ]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: sample

def sample(parameters, char_to_ix, seed):
    """
    根据 RNN 输出的概率分布序列对字符序列进行采样

    参数：
    parameters -- 包含参数 Waa、Wax、Wya、by 和 b 的 Python 字典。 
    char_to_ix -- 将每个字符映射到索引的 Python 字典。
    seed -- 用于评分需要。不用担心。

    返回：
    indices -- 长度为n的列表，包含采样字符的索引。
    """
    
    # 从"parameters"字典中检索参数和相关形状
    Waa, Wax, Wya, by, b = parameters['Waa'], parameters['Wax'], parameters['Wya'], parameters['by'], parameters['b']
    vocab_size = by.shape[0]
    n_a = Waa.shape[1]
    
    ### START CODE HERE ###
    # 步骤 1：创建可用作 one-hot 向量的零向量 x 
    # 代表第一个字符（初始化序列生成）。 （约1行）
    x = None
    # 步骤 1'：将 a_prev 初始化为零（≈1 行）
    a_prev = None
    
    # 创建一个空索引列表。这是包含要生成的字符索引列表的列表（约 1 行）
    indices = []
    
    # idx 是设置为 1 的 one-hot 向量 x 的索引
    # x 中的所有其他位置均为零。
    # 将 idx 初始化为-1
    idx = None 
    
    # 循环时间步 t。在每个时间步：
    # 从概率分布中采样字符 
    # 并将其索引（`idx`）添加到列表"indices"中。 
    # 达到 50 个字符就会停止 
    # （对于训练有素的模型来说这应该是不太可能的）。
    # 设置最大字符数有助于调试并防止无限循环。 
    counter = 0
    newline_character = char_to_ix['\n']
    
    while (idx != newline_character and counter != 50):
        
        # 步骤 2：使用方程 (1)、(2) 和 (3) 前向传播 x
        a = None
        z = None
        y = None
        
        # 用于评分目的
        np.random.seed(counter + seed) 
        
        # 步骤 3：从概率分布 y 中采样词汇表中字符的索引
        # （请参阅上面的其他提示）
        idx = None

        # 将索引附加到"indices"
        indices.append(idx)
        
        # 第 4 步：将输入 x 更新为与采样索引 `idx` 对应的 one-hot 向量。
        # （请参阅上面的其他提示）
        x = None
        x[idx] = None
        
        # 将 "a_prev" 更新为 "a"
        a_prev = None
        
        # 用于评分目的
        seed += 1
        counter +=1
        
    ### END CODE HERE ###

    if (counter == 50):
        indices.append(char_to_ix['\n'])
    
    return indices

In [ ]:
def sample_test(target):
    np.random.seed(24)
    _, n_a = 20, 100
    Wax, Waa, Wya = np.random.randn(n_a, vocab_size), np.random.randn(n_a, n_a), np.random.randn(vocab_size, n_a)
    b, by = np.random.randn(n_a, 1), np.random.randn(vocab_size, 1)
    parameters = {"Wax": Wax, "Waa": Waa, "Wya": Wya, "b": b, "by": by}


    indices = target(parameters, char_to_ix, 0)
    print("Sampling:")
    print("list of sampled indices:\n", indices)
    print("list of sampled characters:\n", [ix_to_char[i] for i in indices])
    
    assert len(indices) < 52, "Indices length must be smaller than 52"
    assert indices[-1] == char_to_ix['\n'], "All samples must end with \\n"
    assert min(indices) >= 0 and max(indices) < len(char_to_ix), f"Sampled indexes must be between 0 and len(char_to_ix)={len(char_to_ix)}"
    assert np.allclose(indices, [23, 16, 26, 26, 24, 3, 21, 1, 7, 24, 15, 3, 25, 20, 6, 13, 10, 8, 20, 12, 2, 0]), "Wrong values"
    
    print("\033[92mAll tests passed!")

sample_test(sample)

**预期输出**
```
Sampling:
list of sampled indices:
 [23, 16, 26, 26, 24, 3, 21, 1, 7, 24, 15, 3, 25, 20, 6, 13, 10, 8, 20, 12, 2, 0]
list of sampled characters:
 ['w', 'p', 'z', 'z', 'x', 'c', 'u', 'a', 'g', 'x', 'o', 'c', 'y', 't', 'f', 'm', 'j', 'h', 't', 'l', 'b', '\n']
```

<div style="color:blue;">

<b>你应该记住什么</b>： 
* 非常大，或者“爆炸式”的梯度更新可能太大，以至于它们在反向传播过程中“超出”最佳值——使训练变得困难 
    * 更新参数前裁剪梯度，以避免梯度爆炸 
* 采样是一种可用于根据概率分布选择下一个字符的索引的技术。
    * 开始字符级采样： 
        * 输入一个由零组成的“虚拟”向量作为默认输入 
        * 运行前向传播的一个步骤，得到𝑎⟨1⟩（你的第一个字符）和𝑦̂ ⟨1⟩（下一个字符的概率分布） 
        * 采样时，使用 `np.random.choice` 避免每次给定起始字母时生成相同的结果（并使你的名字更有趣！）

</div>

<a name='3'></a>
## 3 - 构建语言模型 

是时候构建用于文本生成的字符级语言模型了！ 


<a name='3-1'></a>
### 3.1 - 梯度下降 

在本节中，你将实现函数执行随机梯度下降的一步（带有裁剪的梯度）。你将一次遍历一个训练样本，因此优化算法将是随机的梯度下降。 

提醒一下，以下是 RNN 的常见优化循环的步骤：

- 通过 RNN 前向传播来计算损失
- 通过时间反向传播以计算相对于参数的损失梯度
- 夹住梯度
- 使用梯度下降更新参数 

<a name='ex-3'></a>
### 练习 3 - optimize

实现优化过程（随机梯度下降的一步）。

提供以下函数：

```python
def rnn_forward(X, Y, a_prev, parameters):
    """ 通过RNN执行前向传播并计算交叉熵损失。
    它返回损失值以及 "cache" 存储要在反向传播中使用的值。"""
    ....
    return loss, cache
    
def rnn_backward(X, Y, parameters, cache):
    """ 通过时间执行反向传播来计算损失的梯度
    到参数。它还返回所有隐藏状态。"""
    ...
    return gradients, a

def update_parameters(parameters, gradients, learning_rate):
    """ 使用梯度下降更新规则更新参数。"""
    ...
    return parameters
```

回想一下你之前实现的 `clip` 函数：

#### 参数

* 请注意，`parameters` 字典中的权重和偏置正在通过优化进行更新，即使 `parameters` 不是 `optimize` 函数的返回值之一。 `parameters` 字典通过引用传递到函数，因此即使在函数外部访问，对该字典的更改也会对 `parameters` 字典进行更改。
* Python 字典和列表是“按引用传递”，这意味着如果将字典传递到函数并修改函数中的字典，则会更改同一字典（它不是字典的副本）。

In [ ]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: optimize

def optimize(X, Y, a_prev, parameters, learning_rate = 0.01):
    """
    执行优化的一步来训练模型。
    
    参数：
    X -- 整数列表，其中每个整数都是映射到词汇表中的字符的数字。
    Y -- 整数列表，与 X 完全相同，但向左移动了一个索引。
    a_prev -- 以前的隐藏状态。
    parameters -- Python字典包含：
                        Wax -- 权重矩阵乘以输入，NumPy 形状为 (n_a, n_x) 的数组
                        Waa -- 权重矩阵乘以形状为 (n_a, n_a) 的隐藏状态、NumPy 数组
                        Wya -- 将隐藏状态与输出相关的权重矩阵，形状为 (n_y, n_a) 的 NumPy 数组
                        b --  偏置，NumPy 形状数组 (n_a, 1)
                        by -- 输出层的偏置，将隐藏状态映射至输出时使用，为形状 (n_y, 1) 的 NumPy 数组
    learning_rate -- 模型的学习率。
    
    返回：
    loss -- 损失函数 (cross-entropy) 的值
    gradients -- python 字典包含：
                        dWax -- 输入到隐藏权重的梯度，形状为 (n_a, n_x)
                        dWaa -- 梯度隐藏到隐藏的权重，形状为 (n_a, n_a)
                        dWya -- 隐藏到输出权重的梯度，形状为 (n_y, n_a)
                        db -- 偏置向量梯度，形状为 (n_a, 1)
                        dby -- 输出偏置向量的梯度，形状为 (n_y, 1)
    a[len(X)-1] -- 最后一个隐藏状态，形状为 (n_a, 1)
    """
    
    ### START CODE HERE ###
    
    # 通过时间前向传播（≈1 行代码）
    loss, cache = None
    
    # 时间反向传播（≈1 行）
    gradients, a = None
    
    # 将梯度夹在 -5（最小）和 5（最大）之间（约 1 行代码）
    gradients = None
    
    # 更新参数（约1行） python 中的
    parameters = None
    
    ### END CODE HERE ###
    
    return loss, gradients, a[len(X)-1]

In [ ]:
def optimize_test(target):
    np.random.seed(1)
    vocab_size, n_a = 27, 100
    a_prev = np.random.randn(n_a, 1)
    Wax, Waa, Wya = np.random.randn(n_a, vocab_size), np.random.randn(n_a, n_a), np.random.randn(vocab_size, n_a)
    b, by = np.random.randn(n_a, 1), np.random.randn(vocab_size, 1)
    parameters = {"Wax": Wax, "Waa": Waa, "Wya": Wya, "b": b, "by": by}
    X = [12, 3, 5, 11, 22, 3]
    Y = [4, 14, 11, 22, 25, 26]
    old_parameters = copy.deepcopy(parameters)
    loss, gradients, a_last = target(X, Y, a_prev, parameters, learning_rate = 0.01)
    print("Loss =", loss)
    print("gradients[\"dWaa\"][1][2] =", gradients["dWaa"][1][2])
    print("np.argmax(gradients[\"dWax\"]) =", np.argmax(gradients["dWax"]))
    print("gradients[\"dWya\"][1][2] =", gradients["dWya"][1][2])
    print("gradients[\"db\"][4] =", gradients["db"][4])
    print("gradients[\"dby\"][1] =", gradients["dby"][1])
    print("a_last[4] =", a_last[4])
    
    assert np.isclose(loss, 126.5039757), "Problems with the call of the rnn_forward function"
    for grad in gradients.values():
        assert np.min(grad) >= -5, "Problems in the clip function call"
        assert np.max(grad) <= 5, "Problems in the clip function call"
    assert np.allclose(gradients['dWaa'][1, 2], 0.1947093), "Unexpected gradients. Check the rnn_backward call"
    assert np.allclose(gradients['dWya'][1, 2], -0.007773876), "Unexpected gradients. Check the rnn_backward call"
    assert not np.allclose(parameters['Wya'], old_parameters['Wya']), "parameters were not updated"
    
    print("\033[92mAll tests passed!")

optimize_test(optimize)

**预期输出**

```
Loss = 126.50397572165389
gradients["dWaa"][1][2] = 0.19470931534713587
np.argmax(gradients["dWax"]) = 93
gradients["dWya"][1][2] = -0.007773876032002162
gradients["db"][4] = [-0.06809825]
gradients["dby"][1] = [0.01538192]
a_last[4] = [-1.]
```

<a name='3-2'></a>
### 3.2 - 训练模型 

* 给定恐龙名称数据集，你将使用数据集的每一行（一个名称）作为一个训练样本。 
* 每 2000 步随机梯度下降，你将采样几个随机选择的名称，以查看算法的表现。 
 
<a name='ex-4'></a>
### 练习 4 - model

实现`model()`。 

当 `examples[index]` 包含一个恐龙名称（字符串）时，要创建一个示例（X，Y），可以使用以下命令：

##### 将索引 `idx` 设置到示例列表中

* 使用 for 循环，遍历列表“examples”中经过排序的恐龙名称列表。
* 例如，如果有 n_e 示例，并且 for 循环将索引递增到 n_e 开始，请考虑如何使索引循环回到 0，以便在 j 为 n_e、n_e + 1 等时继续将示例输入到模型中。
* 提示：(n_e + 1) % n_e 等于 1，否则就是 (n_e + 1) 除以 n_e 时得到的“余数”。
* `%` 是 [模运算符](https://www.freecodecamp.org/news/the-python-modulo-operator-what-does-the-symbol-mean-in-python-solved/)。

##### 从示例列表中提取单个示例
* `single_example`：使用你之前设置的 `idx` 索引从示例列表中获取一个单词。

##### 将字符串转换为字符列表：`single_example_chars`
* `single_example_chars`：字符串是字符列表。
* 你可以使用列表理解（推荐使用 for 循环）来生成字符列表。
```Python
str = 'I love learning'
list_of_chars = [c for c in str]
print(list_of_chars)
```

```
['I', ' ', 'l', 'o', 'v', 'e', ' ', 'l', 'e', 'a', 'r', 'n', 'i', 'n', 'g']
```

* 有关 [列表推导式](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions) 的更多信息：

##### 将字符列表转换为整数列表：`single_example_ix`
* 创建一个列表，其中包含与每个字符关联的索引号。
* 使用字典`char_to_ix`
* 你可以将其与用于从字符串中获取字符列表的列表理解结合起来。

##### 创建输入字符列表：`X`
* `rnn_forward` 使用 **`None`** 值作为标志将输入向量设置为零向量。
* 在输入字符*索引*列表前面添加列表[**`None`**]。
* 将值添加到列表的方法不止一种。  一种方法是将两个列表添加在一起：`['a'] + ['b']`

##### 获取换行符`ix_newline`的整数表示
* `ix_newline`：换行符表示恐龙名称的结束。
    - 获取换行符 `'\n'` 的整数表示。
    - 使用`char_to_ix`

##### 设置标签列表（字符的整数表示）：`Y`
* 目标是训练 RNN 预测名称中的下一个字母，因此标签是比输入 `X` 中的字符提前一个时间步的字符列表。
    - 例如，`Y[0]` 包含与 `X[1]` 相同的值  
* RNN 应该在最后一个字母处预测换行符，因此将 `ix_newline` 添加到标签的末尾。 
    - 将换行符的整数表示形式附加到 `Y` 的末尾。
    - 请注意，`append` 是就地操作。
    - 将两个列表添加在一起可能会更容易。

In [ ]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: model

def model(data_x, ix_to_char, char_to_ix, num_iterations = 35000, n_a = 50, dino_names = 7, vocab_size = 27, verbose = False):
    """
    训练模型并生成恐龙名称。 
    
    参数：
    data_x -- 文本语料库，按单词划分
    ix_to_char -- 将索引映射到字符的字典
    char_to_ix -- 将字符映射到索引的字典
    num_iterations -- 训练模型的迭代次数
    n_a -- RNN 单元的单元数
    dino_names -- 你想要在每次迭代中采样的恐龙名称数量。 
    vocab_size -- 文本中找到的唯一字符的数量（词汇量的大小）
    
    返回：
    parameters -- 学习到的参数
    """
    
    # 从 vocab_size 中检索 n_x 和 n_y
    n_x, n_y = vocab_size, vocab_size
    
    # 初始化参数
    parameters = initialize_parameters(n_a, n_x, n_y)
    
    # 初始化损失（这是必需的，因为我们想要平滑我们的损失）
    loss = get_initial_loss(vocab_size, dino_names)
    
    # 构建所有恐龙名称列表（训练样本）。
    examples = [x.strip() for x in data_x]
    
    # 所有恐龙名称的随机列表
    np.random.seed(0)
    np.random.shuffle(examples)
    
    # 初始化LSTM的隐藏状态
    a_prev = np.zeros((n_a, 1))
    
    # 用于评分目的
    last_dino_name = "abc"
    
    # 优化循环
    for j in range(num_iterations):
        
        ### START CODE HERE ###
        
        # 设置索引 `idx` （参见上面的说明）
        idx = None
        
        # 设置输入X（参见上面的说明）
        single_example = None
        single_example_chars = None
        single_example_ix = None
        X = None
        
        # 设置标签 Y（参见上面的说明）
        ix_newline = None
        Y = None

        # 执行一个优化步骤：Forward-prop -> Backward-prop -> Clip -> 更新参数
        # 选择 0.01 的学习率
        curr_loss, gradients, a_prev = None
        
        ### END CODE HERE ###
        
        # 调试语句有助于正确形成 X、Y
        if verbose and j in [0, len(examples) -1, len(examples)]:
            print("j = " , j, "idx = ", idx,) 
        if verbose and j in [0]:
            print("single_example =", single_example)
            print("single_example_chars", single_example_chars)
            print("single_example_ix", single_example_ix)
            print(" X = ", X, "\n", "Y =       ", Y, "\n")
        
        # 保持损失平稳。
        loss = smooth(loss, curr_loss)

        # 每 2000 次迭代，通过 sample() 生成 "n" 字符，以检查模型是否正常学习
        if j % 2000 == 0:
            
            print('Iteration: %d, Loss: %f' % (j, loss) + '\n')
            
            # 要打印的恐龙名称数量
            seed = 0
            for name in range(dino_names):
                
                # 样本索引并打印它们
                sampled_indices = sample(parameters, char_to_ix, seed)
                last_dino_name = get_sample(sampled_indices, ix_to_char)
                print(last_dino_name.replace('\n', ''))
                
                seed += 1  # 要获得相同的结果（出于评分目的），请将种子增加一。 
      
            print('\n')
        
    return parameters, last_dino_name

当你运行以下单元格时，你应该观察到模型在第一次迭代时输出看起来随机的字符。经过几千次迭代后，你的模型应该学会生成看起来合理的名称。 

In [ ]:
parameters, last_name = model(data.split("\n"), ix_to_char, char_to_ix, 22001, verbose = True)

assert last_name == 'Trodonosaurus\n', "Wrong expected output"
print("\033[92mAll tests passed!")

**预期输出**
```
...
Iteration: 22000, Loss: 22.728886

Onustreofkelus
Llecagosaurus
Mystolojmiaterltasaurus
Ola
Yuskeolongus
Eiacosaurus
Trodonosaurus
```

### 结论

你可以看到你的算法在训练即将结束时已开始生成可信的恐龙名称。起初，它生成随机字符，但到最后你可以开始看到带有很酷结尾的恐龙名字。请随意运行算法更长时间并使用超参数，看看是否可以获得更好的结果！我们的实现生成了一些非常酷的名称，例如 `maconucon`、`marloralus` 和 `macingsersaurus`。希望你的模型还了解到恐龙名称往往以 `saurus`、`don`、`aura`、`tor` 等结尾。

如果你的模型生成了一些不酷的名字，不要完全责怪模型——并不是所有真正的恐龙名字听起来都很酷。 （例如，`dromaeosauroides` 是一个真实的恐龙名称，并且位于训练集中。）但是此模型应该为你提供一组候选者，你可以从中选择最酷的！ 

这个作业使用了相对较小的数据集，因此你可以在 CPU 上快速训练 RNN。训练英语模型需要更大的数据集，通常需要更多的计算，并且可能在 GPU 上运行多个小时。我们给恐龙起的名字已经有一段时间了，到目前为止，我们最喜欢的名字是伟大、凶猛、不败的：**芒果龙**！

<img src="images/mangosaurus.jpeg" style="width:250px;height:300px;">

### 恭喜！ 

你已经完成了 Notebook 的评分部分并创建了工作语言模型！很棒的工作。 

到目前为止，你已经： 

* 存储的文本数据，以便使用 RNN 进行处理 
* 构建了字符级文本生成模型
* 探索 RNN 中的消失/爆炸梯度问题
* 应用梯度裁剪以避免梯度爆炸

你还希望生成一些足够酷的恐龙名称来取悦你并避免恐龙的愤怒。如果你对作业感兴趣，请务必不要错过不计分的部分，在这里你将能够像吟游诗人本人一样创作诗歌。祝你好运，玩得开心！

<a name='4'></a>
## 4 - 像莎士比亚一样写作（可选/不计分）

Notebook 的其余部分是可选的，没有评分，但它非常有趣且信息丰富，因此强烈鼓励你尝试一下！ 

与字符级文本生成类似的任务（但更复杂）是生成莎士比亚诗歌。你可以使用莎士比亚诗歌集，而不是从恐龙名称数据集中学习。使用 LSTM 单元，你可以了解跨越文本中许多字符的长期依赖关系 - e.g，其中出现在序列某处的字符可能会影响序列中较晚的字符。这些长期依赖性对于恐龙名称来说不太重要，因为名称很短。 


<img src="images/shakespeare.jpg" style="width:500px;height:400px;">
<div style="text-align:center;color:purple;"><b>让我们成为诗人吧！</b></div>

下面，你可以使用 Keras 实现莎士比亚诗歌生成器。运行以下单元来加载所需的包和模型。这可能需要几分钟。 

In [ ]:
from __future__ import print_function
from tensorflow.keras.callbacks import LambdaCallback
from tensorflow.keras.models import Model, load_model, Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, Input, Masking
from tensorflow.keras.layers import LSTM
from tensorflow.keras.utils import get_file
from tensorflow.keras.preprocessing.sequence import pad_sequences
from shakespeare_utils import *
import sys
import io

为了节省你的时间，模型已经在名为“<i>[十四行诗](shakespeare.txt)</i>”的莎士比亚诗集上训练了约 1000 个 epoch。

再训练模型一个 epoch。训练完成后（同样需要几分钟），运行 `generate_output`，按提示输入文本（`<`40 个字符）。诗歌将以你的句子开头，由你的 RNN“莎士比亚”续写其余部分！例如，可以输入“Forsooth this maketh no sense”（不含引号）。末尾是否包含空格也可能影响结果，因此可以分别尝试，并试用其他输入。 

In [ ]:
print_callback = LambdaCallback(on_epoch_end=on_epoch_end)

model.fit(x, y, batch_size=128, epochs=1, callbacks=[print_callback])

In [ ]:
# 运行此单元以尝试不同的输入，而无需重新训练模型 
generate_output()

### 恭喜你完成了这个Notebook！ 

RNN 莎士比亚模型与你为恐龙名称构建的模型非常相似。唯一的主要区别是：
- LSTM 代替基本的 RNN 来捕获更长范围的依赖关系
- 该模型是更深的堆叠式 LSTM 模型（2 层）
- 使用Keras代替Python来简化代码 


<a name='5'></a>
## 5 - 参考文献 
- 此练习的灵感来自 Andrej Karpathy 的实现：https://gist.github.com/karpathy/d4dee566867f8291f086. 要了解有关文本生成的更多信息，还可以查看 Karpathy 的 [博客文章](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)。